# Synthesis generation audit

Inspect where the best valid candidate appeared, when synthesis first met its recorded convergence criterion, and how much generations 11–20 improved longer runs.

Start with the existing study database or point `DATABASE_PATH` at a separate pilot database. The connection is read-only. The notebook uses recorded candidate errors and convergence flags and does not launch synthesis or benchmark trials.

In [12]:
import os
import sqlite3
from functools import partial
from pathlib import Path
import sys

from IPython.display import display
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker

cwd = Path.cwd()
root = next(path for path in (cwd, *cwd.parents) if (path / "src/evolution").is_dir())
for import_path in (root, root / "src"):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from evolution.application.synthesis.generation_analysis import analyze_generations
from evolution.infra.storage.synthesis.repository import SQLiteSynthesisRepository
from notebooks.analysis.plotting.style import FONT_FAMILY, get_solver_color
from shared.infra.database.tables import ExperimentORM

DATABASE_PATH = Path(os.environ.get("AAD_SYNTHESIS_DB", root / "data/db.sqlite3"))
MODEL_IDS = None  # Optional tuple of exact recorded names, e.g. ("model-id",).

In [13]:
database_path = (root / DATABASE_PATH).resolve()
uri = f"{database_path.as_uri()}?mode=ro"
engine = create_engine("sqlite://", creator=partial(sqlite3.connect, uri, uri=True))
try:
    ids_query = select(ExperimentORM.id).order_by(ExperimentORM.id)
    if MODEL_IDS is not None:
        ids_query = ids_query.where(ExperimentORM.llm_name.in_(MODEL_IDS))
    with engine.connect() as connection:
        experiment_ids = list(connection.execute(ids_query).scalars())

    repository = SQLiteSynthesisRepository(sessionmaker(engine))
    analysis = analyze_generations(repository.load_by_ids(experiment_ids))
finally:
    engine.dispose()

print(f"Database: {database_path}")
print(f"Recorded experiments selected: {len(experiment_ids)}")

Database: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/db.sqlite3
Recorded experiments selected: 2079


## When did experiments find their best candidate?

One **experiment** (also called a run) is one LLaMEA attempt at a fixed problem and configuration. A **generation** is one candidate attempt within that experiment. Each percentage counts experiments, with the denominator shown on the chart.

The bars answer **“At which generation did the best candidate first appear?”** The lines answer **“How many experiments had reached this outcome by that generation?”** For example, “by generation 3” includes generations 1, 2, and 3. Best candidate means the lowest error observed within the experiment. Reaching the convergence threshold is tracked separately using the recorded convergence flag.


In [14]:
ten = analysis.distribution[
    (analysis.distribution["generations"] == 10) & analysis.distribution["model"].isna()
]

if ten.empty:
    print("No completed 10-generation experiments with all generations recorded.")
else:
    totals = analysis.summary[
        (analysis.summary["generations"] == 10) & analysis.summary["model"].isna()
    ].iloc[0]
    valid = int(totals["valid"])
    eligible = int(totals["experiments"])
    example = ten.loc[ten["generation"] == 3].iloc[0]
    generation_figure = make_subplots(
        rows=3,
        cols=1,
        vertical_spacing=0.14,
        subplot_titles=(
            f"Best candidate first appeared at this generation · {valid:,} experiments",
            f"Best candidate already found by this generation · {valid:,} experiments",
            f"Convergence threshold reached by this generation · {eligible:,} experiments",
        ),
    )
    generation_figure.add_bar(
        x=ten["generation"],
        y=ten["best_fraction"],
        customdata=ten["best_count"],
        text=ten["best_fraction"],
        texttemplate="%{text:.1%}",
        textposition="outside",
        marker_color=["#315d89"] * 9 + ["#b94e21"],
        hovertemplate=(
            "At generation %{x}<br>%{customdata:,} of "
            f"{valid:,} experiments (%{{y:.1%}})<extra></extra>"
        ),
        row=1,
        col=1,
    )
    for row, column, count_column, denominator, color in (
        (2, "cumulative_best_fraction", "best_count", valid, "#315d89"),
        (3, "cumulative_converged_fraction", "converged_count", eligible, "#16826d"),
    ):
        generation_figure.add_scatter(
            x=ten["generation"],
            y=ten[column],
            customdata=ten[count_column].cumsum(),
            mode="lines+markers",
            line=dict(color=color, width=3, shape="hv"),
            hovertemplate=(
                "By generation %{x}<br>%{customdata:,} of "
                f"{denominator:,} experiments (%{{y:.1%}})<extra></extra>"
            ),
            row=row,
            col=1,
        )
        generation_figure.update_yaxes(range=[0, 1.05], row=row, col=1)
    if valid:
        generation_figure.add_annotation(
            x=3,
            y=example["cumulative_best_fraction"],
            text=f"By generation 3: {example['cumulative_best_fraction']:.1%}",
            arrowhead=2,
            ax=85,
            ay=-28,
            bgcolor="white",
            row=2,
            col=1,
        )
    generation_figure.update_xaxes(
        title_text="LLaMEA generation", dtick=1, range=[0.5, 10.5], fixedrange=True
    )
    generation_figure.update_yaxes(title_text="% of experiments", tickformat=".0%")
    generation_figure.update_yaxes(
        range=[0, max(0.05, ten["best_fraction"].max() * 1.22)] if valid else [0, 1],
        row=1,
        col=1,
    )
    generation_figure.update_layout(
        title=dict(
            text=(
                "When did 10-generation experiments get their results?"
                "<br><sup>Best-candidate charts include experiments with a finite score. "
                "The convergence chart includes every eligible experiment.</sup>"
            ),
            x=0.02,
        ),
        height=855,
        margin=dict(t=110, b=55),
        template="plotly_white",
        font_family=FONT_FAMILY,
        showlegend=False,
    )
    display(generation_figure)


## Did generations 11–20 improve the result?

Each dot represents one completed 20-generation experiment with a valid score at both cutoffs. On the left, dots below the diagonal improved. On the right, a larger positive difference means a larger drop in error.


In [15]:
pairs = analysis.pairs.dropna(subset=["error_at_10", "error_at_20"])

if pairs.empty:
    print("No completed 20-generation experiments have valid scores at both cutoffs.")
else:
    paired_figure = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Best error at each cutoff", "Absolute improvement"),
    )
    for model, group in pairs.groupby("model", sort=True):
        color = get_solver_color(model)
        paired_figure.add_scatter(
            x=group["error_at_10"],
            y=group["error_at_20"],
            text=group["experiment_id"],
            hovertemplate="Experiment %{text}<br>10: %{x:.4g}<br>20: %{y:.4g}<extra></extra>",
            mode="markers",
            marker_color=color,
            name=model,
            legendgroup=model,
            row=1,
            col=1,
        )
        paired_figure.add_scatter(
            x=group["experiment_id"],
            y=group["absolute_improvement"],
            text=group["experiment_id"],
            hovertemplate="Experiment %{text}<br>Improvement: %{y:.4g}<extra></extra>",
            name=model,
            marker_color=color,
            legendgroup=model,
            showlegend=False,
            mode="markers",
            row=1,
            col=2,
        )
    error_limit = pairs["error_at_10"].max()
    paired_figure.add_shape(
        type="line", x0=0, y0=0, x1=error_limit, y1=error_limit,
        line=dict(color="#64748B", dash="dot"), row=1, col=1,
    )
    paired_figure.update_xaxes(title_text="Best error at generation 10", row=1, col=1)
    paired_figure.update_yaxes(title_text="Best error at generation 20", row=1, col=1)
    paired_figure.update_xaxes(title_text="Experiment ID", row=1, col=2)
    paired_figure.update_yaxes(title_text="Error at 10 minus error at 20", row=1, col=2)
    pooled = analysis.paired_summary.loc[analysis.paired_summary["model"].isna()].iloc[0]
    paired_figure.update_layout(
        title=(
            "How much did generations 11–20 help?"
            f"<br><sup>{pooled['improved_fraction']:.1%} improved "
            f"({pooled['improved']:,} of {pooled['comparable']:,} comparable experiments)"
            "</sup>"
        ),
        height=490,
        template="plotly_white",
        font_family=FONT_FAMILY,
    )
    display(paired_figure)

display(analysis.pairs)

No completed 20-generation experiments have valid scores at both cutoffs.


,experiment_id,model,error_at_10,error_at_20,absolute_improvement,relative_improvement,best_after_10,first_convergence_after_10


## Experiment details and exclusions

Inspect the per-run outcomes and the records excluded from these comparisons using their experiment IDs. Model names and seeds are taken from the database.

In [16]:
display(analysis.summary)
display(
    analysis.paired_summary.style.format(
        {"improved_fraction": "{:.1%}", "median_relative_improvement": "{:.1%}"},
        na_rep="—",
    )
)
display(analysis.pairs)
display(analysis.runs)
display(analysis.exclusions)


,model,generations,experiments,valid,converged,median_best_generation,best_after_10,first_convergence_after_10
0,NaN,10,1434,1434,452,5.0,0,0
1,gwdg.qwen3-coder-next,10,477,477,149,3.0,0,0
2,openai.gpt-6-luna,10,49,49,32,2.0,0,0
3,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,462,462,119,6.0,0,0
4,qwen2.5-coder-32b-instruct-q4_k_m.gguf,10,446,446,152,5.0,0,0


,model,experiments,valid_prefix,missing_valid_prefix,valid_full_run,comparable,improved,improved_fraction,relative_defined,median_absolute_improvement,median_relative_improvement
0,—,0,0,0,0,0,0,—,0,—,—


,experiment_id,model,error_at_10,error_at_20,absolute_improvement,relative_improvement,best_after_10,first_convergence_after_10


,experiment_id,model,generations,status,observed_generations,synthesis_seed,problem_id,dim,instance_id,noise_std,noise_model,mode,strategy,budget,best_generation,best_error,first_convergence_generation
0,5339,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,10,43,1,50,1,0.0,none,explicit,baseline,100000,4,4.663139e-06,NaN
1,5342,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,10,43,1,50,1,0.0,none,explicit,guided,100000,7,2.842171e-14,1.0
2,5343,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,10,43,1,50,1,0.0,none,implicit,baseline,100000,8,6.694833e+01,NaN
3,5344,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,10,43,1,50,1,0.0,none,implicit,thinking,100000,10,8.834819e-02,NaN
4,5345,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,10,43,1,50,1,0.0,none,implicit,vectorization,100000,10,4.027220e-03,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1429,7366,openai.gpt-6-luna,10,completed,10,43,21,50,1,0.2,heteroscedastic,explicit,baseline,100000,9,7.487586e+01,NaN
1430,7367,openai.gpt-6-luna,10,completed,10,43,21,50,1,0.2,heteroscedastic,explicit,vectorization,100000,2,7.076479e+01,NaN
1431,7368,openai.gpt-6-luna,10,completed,10,43,1,50,1,0.0,none,explicit,baseline,100000,1,1.421085e-14,1.0
1432,7373,openai.gpt-6-luna,10,completed,10,43,1,50,1,0.2,heteroscedastic,implicit,vectorization,100000,1,2.842171e-14,1.0


,experiment_id,model,generations,status,observed_generations,synthesis_seed,problem_id,dim,instance_id,noise_std,noise_model,mode,strategy,budget,reason
0,5340,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,9,43,1,50,1,0.0,none,explicit,thinking,100000,iteration_count_mismatch
1,5341,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,9,43,1,50,1,0.0,none,explicit,vectorization,100000,iteration_count_mismatch
2,5470,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,9,43,21,50,1,0.0,none,explicit,guided,100000,iteration_count_mismatch
3,5485,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,9,43,21,50,1,0.1,heteroscedastic,explicit,vectorization,100000,iteration_count_mismatch
4,5691,qwen2.5-coder-14b-instruct-q4_k_m.gguf,10,completed,9,43,15,2,1,0.0,none,explicit,baseline,100000,iteration_count_mismatch
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,7413,openai.gpt-6-luna,10,running,0,43,15,50,1,0.0,none,implicit,vectorization,100000,not_completed
641,7414,openai.gpt-6-luna,10,running,0,43,21,50,1,0.0,none,explicit,baseline,100000,not_completed
642,7415,openai.gpt-6-luna,10,running,0,43,21,50,1,0.0,none,explicit,vectorization,100000,not_completed
643,7416,openai.gpt-6-luna,10,running,0,43,21,50,1,0.0,none,implicit,baseline,100000,not_completed
